# 문항 1 : 네이버 연관검색어 수집 함수 만들기

## 키워드를 입력받아 그 키워드의 연관검색어 리스트를 돌려주는 함수를 작성하시오.

함수명: get_related_keywords(keyword)

반환: 문자열 리스트

Selenium 사용 금지. Network 탭에서 요청을 찾아 requests로 재현할 것

결과가 없으면 빈 리스트를 돌려줄 것 (예외를 던지지 말 것)

-- 힌트.md에 문서 찾기 힌트, 결과 파싱 힌트 있음 --

### 실행 예시

>>>get_related_keywords('부트캠프')
['부트캠프 뜻', '부트캠프', '부트캠프 추천', '카카오 부트캠프', '맥북 부트캠프',
 '카카오테크 부트캠프', '직무부트캠프', '네이버 부트캠프', '마케팅 부트캠프', '코멘토 직무부트캠프']

In [1]:
import requests

In [ ]:
def get_related_keywords(keyword):
    result = []

    URL = "https://ac.search.naver.com/nx/ac?"

    headers = {
        "User-Agent": 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
    }

    PARAMS = {
        "q": keyword,
        "q_enc": 'UTF-8',
        "st": '100',
        "r_format": 'json',
        "r_enc": 'UTF-8',
    }

    try:
        response = requests.get(URL, params=PARAMS, headers=headers)

        if response.status_code == 200:
            data = response.json()

            if 'items' in data and len(data['items']) > 0:
                values = data['items'][0]
                result = [item[0] for item in values if item and len(item) > 0]
    except Exception:
        return []

    return result

In [9]:
if __name__ == "__main__":
    result_ex = get_related_keywords('AI')
    print(result_ex)

['ai', 'ai 판독기', 'ai 뜻', 'ai 대화', 'ai 이미지 생성', 'ai 종류', 'Ai 안경', '제타 ai', 'ai 검사기', '클로드 ai']


# 문항 2 : 네이버 웹툰 전체 목록 수집

## 네이버 웹툰의 요일별 전체 웹툰을 수집하시오.

대상: https://comic.naver.com/webtoon

추출 필드: 제목 / 링크 / 요일 (딕셔너리 형태)

모든 요일의 웹툰을 수집할 것

링크는 상세 페이지로 바로 이동할 수 있는 절대 주소로 만들 것 (힌트 1)

결과를 naver_webtoon.csv로 저장할 것

### 결과 예시

[{'제목': '광마회귀', '링크': 'https://comic.naver.com/webtoon/list?titleId=776601', '요일': '금'},
 {'제목': '외모지상주의', '링크': 'https://comic.naver.com/webtoon/list?titleId=641253', '요일': '금'},
 ...]


In [25]:
import csv
import json
import re
import requests
from bs4 import BeautifulSoup

In [37]:
URL_webtoon = "https://comic.naver.com/api/webtoon/titlelist/weekday?"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
}

PARAMS = {
    "order": "user"
}

def fetch_all() -> list[dict]:
    try:
        res = requests.get(URL_webtoon, headers=headers, params=PARAMS)
        if res.status_code == 200:
            return res.json()
    except Exception as e:
        print(f"오류발생: {e}")
    return None

def parse(data):
    parsed_list = []
    if not data or 'titleListMap' not in data:
        return parsed_list

    title_list_map = data['titleListMap']
    
    weekday_mapping = {
        "MONDAY": "월", "TUESDAY": "화", "WEDNESDAY": "수", "THURSDAY": "목",
        "FRIDAY": "금", "SATURDAY": "토", "SUNDAY": "일", "DAILY_PLUS": "매일+",
    }

    for weekday_code, webtoons in title_list_map.items():
        kor_weekday = weekday_mapping.get(weekday_code, weekday_code)

        for toon in webtoons:
            title = toon.get('titleName')
            title_id = toon.get('titleId')

            if title and title_id:
                absolute_link = f"https://comic.naver.com/webtoon/list?titleId={title_id}"

                webtoon_info = {
                    '제목': title,
                    '링크': absolute_link,
                    '요일': kor_weekday
                }
                parsed_list.append(webtoon_info)

    return parsed_list

if __name__ == "__main__":
    raw_data = fetch_all()
    results = parse(raw_data)

    print(f"최종 수집된 전체 웹툰 개수: {len(results)}개")

최종 수집된 전체 웹툰 개수: 778개


In [38]:
if results:
    filename = "naver_webtoon.csv"
    with open(filename, 'w', encoding='utf-8-sig', newline='') as f:
        fieldnames = ['제목', '링크', '요일']
        writer = csv.DictWriter(f, fieldnames=fieldnames)
            
        writer.writeheader()        # 1행 컬럼명 생성
        writer.writerows(results)   # 추출 리스트 한 번에 밀어넣기
            
    print(f"성공적으로 '{filename}' 저장")

성공적으로 'naver_webtoon.csv' 저장


# 문항 3 :사람인 채용공고 10페이지 수집

## 사람인의 공개 채용공고 목록을 10페이지 수집하시오.

대상: https://www.saramin.co.kr/zf_user/jobs/public/list

추출 필드: 기업명 / 그룹사 / 기업종류 / 공고명 / 직무키워드 / 학력 / 경력구분 / 근무지

직무키워드는 리스트로 담을 것

값이 없는 항목은 빈 문자열 또는 빈 리스트로 둘 것 (오류로 멈추지 말 것)

요청 간 0.5초 이상 지연을 둘 것

결과를 saramin.csv로 저장할 것

### 결과 예시

[{'기업명': '(주)유니드',
  '그룹사': '오씨아이그룹',
  '기업종류': '대기업',
  '공고명': '2025년 유니드 상반기 신입사원 수시채용',
  '직무키워드': ['외환관리', '자금관리', '자산운용', '재무제표', '재무회계'],
  '학력': '대학교(4년)↑',
  '경력구분': '신입 · 정규직',
  '근무지': '서울 중구 외'},
 ...]


In [81]:
URL_job = "https://www.saramin.co.kr/zf_user/jobs/public/list"

def text_of(box, selector):
    if not box:
        return ""
    elem = box.select_one(selector)
    return elem.get_text(strip=True) if elem else ""

def fetch(page):
    res = requests.get(URL_job, headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Whale/4.39.410.14 Safari/537.36",
        "Referer": "https://www.saramin.co.kr/"
    }, params={
        "page": page
    })
    
    if res.status_code == 200:
        return res.text
    else:
        print(f"{page}페이지 요청 실패 (상태코드: {res.status_code})")
    return None

def parse(data: dict) -> list[dict]:
    html = data if isinstance(data, str) else data.get('innerHTML', '')
    soup = BeautifulSoup(html, "html.parser")

    job_list = []
    for box in soup.select("div.list_body div.list_item"):
        keywords = [k.text.strip() for k in box.select("span.job_sector span")]
        job_list.append({
            "기업명": text_of(box, "a.str_tit"),
            "그룹사": text_of(box, "span.main_corp"),
            "기업종류": text_of(box, "span.info_stock"),
            "공고명": text_of(box, "div.notification_info a.str_tit"),
            "직무키워드": keywords,
            "학력": text_of(box, "p.education"),
            "경력구분": text_of(box, "p.career"),
            "근무지": text_of(box, "p.work_place"),
        })
    return job_list

import time
job_list_final = []
for page in range(1, 11):
    datas = fetch(page)
    result = parse(datas)

    if result: 
        job_list_final.extend(result)
    else:
        print(f"오류")

    time.sleep(0.5)


print(f"\n최종 총 수집된 공고 개수: {len(job_list_final)}개")



최종 총 수집된 공고 개수: 200개


In [ ]:
if job_list_final:
    filename = "saramin.csv"
    with open(filename, 'w', encoding='utf-8-sig', newline='') as ff:
        fieldnames = ['기업명', '그룹사', '기업종류', '공고명', '직무키워드', '학력', '경력구분', '근무지']
        writer = csv.DictWriter(ff, fieldnames=fieldnames)
            
        writer.writeheader()        
        writer.writerows(job_list_final)  
            
    print(f"성공적으로 '{filename}' 저장")

성공적으로 'saramin.csv' 저장
